In [1]:
"""
Gold Layer Validation Script
Motor Insurance Quote-to-Policy Conversion

Consolidates every check flagged during the bronze -> silver -> gold review
into one script that reports PASS / FAIL / WARN, instead of re-discovering
the same bugs by eyeballing dashboard output.
Run this after any pipeline rebuild, before touching Power BI or the demo.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import sys

GOLD = Path("../data/gold")
FACT = GOLD / "facts"
DIM = GOLD / "dimensions"
MART = GOLD / "data_marts"

results = []  # (section, check_name, status, detail)


def check(section, name, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    results.append((section, name, status, detail))


def warn(section, name, condition, detail=""):
    status = "PASS" if condition else "WARN"
    results.append((section, name, status, detail))


def safe_read(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        return None


# ==========================================================
# Load everything up front — missing files become their own
# failed checks rather than crashing the whole script
# ==========================================================

tables = {
    "dim_customer": DIM / "dim_customer.csv",
    "dim_vehicle": DIM / "dim_vehicle.csv",
    "dim_policy": DIM / "dim_policy.csv",
    "dim_channel": DIM / "dim_channel.csv",
    "dim_date": DIM / "dim_date.csv",
    "fact_quote": FACT / "fact_quote.csv",
    "fact_claim": FACT / "fact_claim.csv",
    "fact_underwriting": FACT / "fact_underwriting.csv",
    "fact_customer_journey": FACT / "fact_customer_journey.csv",
    "fact_ai_interaction": FACT / "fact_ai_interaction.csv",
    "executive_dashboard": MART / "executive_dashboard.csv",
    "sales_dashboard": MART / "sales_dashboard.csv",
    "underwriting_dashboard": MART / "underwriting_dashboard.csv",
    "claims_dashboard": MART / "claims_dashboard.csv",
    "customer360_dashboard": MART / "customer360_dashboard.csv",
    "ai_monitoring_dashboard": MART / "ai_monitoring_dashboard.csv",
}

dfs = {name: safe_read(path) for name, path in tables.items()}

for name, df in dfs.items():
    check("0. FILE PRESENCE", f"{name}.csv exists and loads", df is not None,
          "" if df is not None else f"missing at {tables[name]}")

# Stop early if the core tables aren't even present — everything below
# assumes they exist
core = ["dim_customer", "dim_vehicle", "dim_policy", "dim_date",
        "fact_quote", "fact_claim", "fact_underwriting"]
if any(dfs[t] is None for t in core):
    print("Core tables missing — fix file presence issues above before rerunning.")
    for r in results:
        print(r)
    sys.exit(1)

customer, vehicle, policy, channel, date = (
    dfs["dim_customer"], dfs["dim_vehicle"], dfs["dim_policy"],
    dfs["dim_channel"], dfs["dim_date"]
)
quote, claim, uw, journey, ai = (
    dfs["fact_quote"], dfs["fact_claim"], dfs["fact_underwriting"],
    dfs["fact_customer_journey"], dfs["fact_ai_interaction"]
)


# ==========================================================
# A. STRUCTURAL — the aligned-population assumption everything
# else depends on
# ==========================================================

n = len(customer)
check("A. STRUCTURAL", "dim_customer/dim_vehicle/dim_policy same row count",
      len(vehicle) == n and len(policy) == n,
      f"customer={n}, vehicle={len(vehicle)}, policy={len(policy)}")

check("A. STRUCTURAL", "fact_quote/fact_claim/fact_underwriting same row count",
      len(quote) == n and len(claim) == n and len(uw) == n,
      f"n={n}, quote={len(quote)}, claim={len(claim)}, uw={len(uw)}")

for name, df, col in [("dim_customer", customer, "customer_sk"),
                       ("dim_vehicle", vehicle, "vehicle_sk"),
                       ("dim_policy", policy, "policy_sk")]:
    check("A. STRUCTURAL", f"{name}.{col} is unique", df[col].is_unique)

# Orphan foreign key checks — catches broken merges before they surface
# as blank rows on a dashboard
fk_checks = [
    ("fact_quote.customer_sk -> dim_customer", quote["customer_sk"], customer["customer_sk"]),
    ("fact_quote.vehicle_sk -> dim_vehicle", quote["vehicle_sk"], vehicle["vehicle_sk"]),
    ("fact_quote.policy_sk -> dim_policy", quote["policy_sk"], policy["policy_sk"]),
    ("fact_claim.customer_sk -> dim_customer", claim["customer_sk"], customer["customer_sk"]),
    ("fact_claim.vehicle_sk -> dim_vehicle", claim["vehicle_sk"], vehicle["vehicle_sk"]),
    ("fact_underwriting.customer_sk -> dim_customer", uw["customer_sk"], customer["customer_sk"]),
    ("fact_underwriting.vehicle_sk -> dim_vehicle", uw["vehicle_sk"], vehicle["vehicle_sk"]),
]
if channel is not None:
    fk_checks.append(("fact_quote.channel_sk -> dim_channel", quote["channel_sk"], channel["channel_sk"]))

for label, child, parent in fk_checks:
    orphans = set(child.dropna()) - set(parent.dropna())
    check("A. STRUCTURAL", f"No orphan keys: {label}", len(orphans) == 0,
          f"{len(orphans)} orphaned values" if orphans else "")


# ==========================================================
# B. DATE INTEGRITY — the independent-random-date bug pattern
# found repeatedly across fact_claim, fact_underwriting,
# fact_customer_journey
# ==========================================================

if date is not None:
    valid_date_sks = set(date["date_sk"])
    date_col_checks = [
        ("fact_quote.date_sk", quote["date_sk"]),
        ("fact_claim.date_sk", claim["date_sk"]),
        ("fact_underwriting.underwriting_date_sk", uw["underwriting_date_sk"]),
    ]
    if journey is not None:
        date_col_checks.append(("fact_customer_journey.date_sk", journey["date_sk"]))
    if ai is not None:
        date_col_checks.append(("fact_ai_interaction.date_sk", ai["date_sk"]))

    for label, col in date_col_checks:
        out_of_range = set(col.dropna()) - valid_date_sks
        check("B. DATES", f"{label} within dim_date range", len(out_of_range) == 0,
              f"{len(out_of_range)} out-of-range values" if out_of_range else "")

    # Events shouldn't predate their originating quote
    quote_dates = quote[["customer_sk", "date_sk"]].rename(columns={"date_sk": "quote_date_sk"})

    claim_check = claim.merge(quote_dates, on="customer_sk", how="left")
    before_quote = (claim_check["date_sk"] < claim_check["quote_date_sk"]).sum()
    check("B. DATES", "fact_claim events occur on/after their quote date",
          before_quote == 0, f"{before_quote} claims dated before their quote")

    uw_check = uw.merge(quote_dates, on="customer_sk", how="left")
    uw_before = (uw_check["underwriting_date_sk"] < uw_check["quote_date_sk"]).sum()
    check("B. DATES", "fact_underwriting events occur on/after their quote date",
          uw_before == 0, f"{uw_before} underwriting rows dated before their quote")

    if ai is not None:
        ai_check = ai.merge(quote_dates, on="customer_sk", how="left")
        ai_before = (ai_check["date_sk"] < ai_check["quote_date_sk"]).sum()
        check("B. DATES", "fact_ai_interaction events occur on/after their quote date",
              ai_before == 0, f"{ai_before} interactions dated before their quote")


# ==========================================================
# C. FUNNEL LOGIC — accepted_offer vs conversion_flag, the
# fix applied a few turns ago
# ==========================================================

check("C. FUNNEL", "conversion_flag never exceeds accepted_offer (row-level)",
      (quote["conversion_flag"] <= quote["accepted_offer"]).all())

accepted_rate = quote["accepted_offer"].mean()
converted_rate = quote["conversion_flag"].mean()
check("C. FUNNEL", "conversion_flag mean is strictly lower than accepted_offer mean",
      converted_rate < accepted_rate,
      f"accepted={accepted_rate:.3f}, converted={converted_rate:.3f}")

if journey is not None:
    max_stage = journey["stage_sequence"].max()
    converted_customers = quote.loc[quote["conversion_flag"] == 1, "customer_sk"]
    last_stage_per_customer = journey.groupby("customer_sk")["stage_sequence"].max()
    converted_reach_end = last_stage_per_customer.reindex(converted_customers).eq(max_stage).all()
    check("C. FUNNEL", "Every converted customer's journey reaches the final stage",
          bool(converted_reach_end))


# ==========================================================
# D. VALUE RANGES / DISTRIBUTIONS — catches degenerate bins
# and scale mismatches (safety_score, quote_value_band, etc.)
# ==========================================================

check("D. RANGES", "dim_vehicle.safety_score within 0-100",
      vehicle["safety_score"].between(0, 100).all(),
      f"min={vehicle['safety_score'].min()}, max={vehicle['safety_score'].max()}")

if "quote_value_band" in quote.columns:
    band_counts = quote["quote_value_band"].value_counts(normalize=True)
    thin_bands = band_counts[band_counts < 0.01]
    warn("D. RANGES", "quote_value_band has no near-empty buckets",
         thin_bands.empty, f"buckets under 1%: {dict(thin_bands.round(3))}" if not thin_bands.empty else "")

if ai is not None:
    conf_std = ai["confidence_score"].std()
    warn("D. RANGES", "fact_ai_interaction.confidence_score has real spread (not artificially floored)",
         conf_std > 0.05, f"std={conf_std:.3f}")

override_rate = (uw["underwriting_decision"] != uw["recommendation"]).mean()
warn("D. RANGES", "underwriting_decision diverges from recommendation sometimes (AI override signal exists)",
     override_rate > 0, f"override_rate={override_rate:.3f}")


# ==========================================================
# E. NULLS / SUSPICIOUS VALUES
# ==========================================================

check("E. NULLS", "fact_claim.claim_flag has no missing values",
      claim["claim_flag"].isna().sum() == 0,
      f"{claim['claim_flag'].isna().sum()} missing")

if dfs["executive_dashboard"] is not None:
    ed = dfs["executive_dashboard"]
    if "claim_ratio" in ed.columns:
        insane_ratio = (ed["claim_ratio"].dropna() > 1000).sum()
        check("E. NULLS", "executive_dashboard.claim_ratio has no runaway values (>1000%)",
              insane_ratio == 0, f"{insane_ratio} rows above 1000% — check the /replace(0,1) pattern")


# ==========================================================
# F. MART-SPECIFIC — regressions we've hit before, checked
# directly against the marts this time
# ==========================================================

if dfs["sales_dashboard"] is not None:
    sd = dfs["sales_dashboard"]
    if "policy_issued_flag" in sd.columns and "accepted_offer" in sd.columns:
        check("F. MARTS", "sales_dashboard.policy_issued_flag tracks conversion_flag, not accepted_offer",
              sd["policy_issued_flag"].mean() < sd["accepted_offer"].mean(),
              f"policy_issued={sd['policy_issued_flag'].mean():.3f}, accepted={sd['accepted_offer'].mean():.3f}")

if dfs["claims_dashboard"] is not None:
    cd = dfs["claims_dashboard"]
    forbidden_customer_cols = {"gender", "customer_age", "region", "customer_segment", "risk_profile"}
    present = forbidden_customer_cols & set(cd.columns)
    check("F. MARTS", "claims_dashboard contains no customer demographic columns",
          len(present) == 0, f"found: {present}" if present else "")

if dfs["customer360_dashboard"] is not None:
    c360 = dfs["customer360_dashboard"]
    check("F. MARTS", "customer360_dashboard row count matches fact_quote",
          len(c360) == len(quote), f"customer360={len(c360)}, fact_quote={len(quote)}")
    if "converted_customer" in c360.columns:
        check("F. MARTS", "customer360_dashboard.converted_customer tracks conversion_flag",
              c360["converted_customer"].mean() == converted_rate,
              f"mart={c360['converted_customer'].mean():.3f}, fact_quote={converted_rate:.3f}")

if dfs["ai_monitoring_dashboard"] is not None:
    amd = dfs["ai_monitoring_dashboard"]
    check("F. MARTS", "ai_monitoring_dashboard does not contain the dropped customer_intent column",
          "customer_intent" not in amd.columns)

old_customer_dashboard = MART / "customer_dashboard.csv"
warn("F. MARTS", "Old customer_dashboard.csv has been removed (merged into customer360)",
     not old_customer_dashboard.exists(),
     f"still present at {old_customer_dashboard}" if old_customer_dashboard.exists() else "")


# ==========================================================
# Report
# ==========================================================

print("=" * 90)
print("GOLD LAYER VALIDATION REPORT")
print("=" * 90)

current_section = None
fail_count = warn_count = pass_count = 0

for section, name, status, detail in results:
    if section != current_section:
        print(f"\n--- {section} ---")
        current_section = section
    marker = {"PASS": "✓", "FAIL": "✗ FAIL", "WARN": "! WARN"}[status]
    line = f"  [{marker}] {name}"
    if detail and status != "PASS":
        line += f"  ({detail})"
    print(line)
    if status == "PASS":
        pass_count += 1
    elif status == "WARN":
        warn_count += 1
    else:
        fail_count += 1

print("\n" + "=" * 90)
print(f"SUMMARY: {pass_count} passed, {warn_count} warnings, {fail_count} failed")
print("=" * 90)

if fail_count > 0:
    print("\nFix FAIL items before building dashboards or the demo on top of this data.")
    sys.exit(1)
elif warn_count > 0:
    print("\nNo blocking failures. Review WARN items — they won't crash anything but may weaken the story.")
else:
    print("\nAll checks passed. Gold layer is clean.")

GOLD LAYER VALIDATION REPORT

--- 0. FILE PRESENCE ---
  [✓] dim_customer.csv exists and loads
  [✓] dim_vehicle.csv exists and loads
  [✓] dim_policy.csv exists and loads
  [✓] dim_channel.csv exists and loads
  [✓] dim_date.csv exists and loads
  [✓] fact_quote.csv exists and loads
  [✓] fact_claim.csv exists and loads
  [✓] fact_underwriting.csv exists and loads
  [✓] fact_customer_journey.csv exists and loads
  [✓] fact_ai_interaction.csv exists and loads
  [✓] executive_dashboard.csv exists and loads
  [✓] sales_dashboard.csv exists and loads
  [✓] underwriting_dashboard.csv exists and loads
  [✓] claims_dashboard.csv exists and loads
  [✓] customer360_dashboard.csv exists and loads
  [✓] ai_monitoring_dashboard.csv exists and loads

--- A. STRUCTURAL ---
  [✓] dim_customer/dim_vehicle/dim_policy same row count
  [✓] fact_quote/fact_claim/fact_underwriting same row count
  [✓] dim_customer.customer_sk is unique
  [✓] dim_vehicle.vehicle_sk is unique
  [✓] dim_policy.policy_sk is u